# PRD_GOLD Synthetic Data Generator

Generates realistic health-plan data from **Apr 2023 through the current date** (dynamic end date) using **[dbldatagen](https://github.com/databrickslabs/dbldatagen)** (Databricks Labs Data Generator).

## How to adjust volume
Edit the **Row Counts** section in the *Configuration* cell:

| Variable | Default | Controls |
|---|---|---|
| `NUM_MEMBERS` | 5,000 | dim_member |
| `NUM_PROVIDERS` | 500 | dim_provider |
| `NUM_ADDRESSES` | members + providers + 1,000 | dim_address |
| `NUM_HISTORY_VERSIONS` | members × 2 | dim_member_history |
| `NUM_MEMBER_IDS` | members × 2 | dim_member_identifier |
| `NUM_ENROLLMENTS` | 10,000 | fact_member_enrollment |
| `NUM_CLAIM_HEADERS` | 50,000 | fact_claim_header |
| `NUM_CLAIM_DETAILS` | 150,000 | fact_claim_detail |

Designed for downstream **AI/BI Dashboards**, **Metric Views**, and **Genie Spaces**.

In [0]:
%pip install dbldatagen --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import dbldatagen as dg
from pyspark.sql.types import *
from pyspark.sql import functions as F
from datetime import date

# ---------------------------------------------------------------------------
# Target catalog / schema  (must match DDL notebook)
# ---------------------------------------------------------------------------
catalog = "aira_test"
schema  = "aibi_workshop_schema_day2"

# ---------------------------------------------------------------------------
# Date range: dynamic end date (always generates data up to today)
# ---------------------------------------------------------------------------
DATA_START = "2023-04-01"
DATA_END   = date.today().strftime("%Y-%m-%d")  # dynamic: always current date

# =====================================================================
#  ROW COUNTS  – adjust these knobs to scale the dataset up or down
# =====================================================================
NUM_MEMBERS          = 20_000
NUM_PROVIDERS        = 1000
NUM_ADDRESSES        = NUM_MEMBERS + NUM_PROVIDERS + 1_000   # members + providers + facilities
NUM_HISTORY_VERSIONS = NUM_MEMBERS * 2       # ~2 history rows per member on avg
NUM_MEMBER_IDS       = NUM_MEMBERS * 2       # ~2 alternate IDs per member
NUM_ENROLLMENTS      = 20_000
NUM_CLAIM_HEADERS    = 100_000
NUM_CLAIM_DETAILS    = 300_000

SPARK_PARTITIONS = 8   # parallelism for data generation

print(f"Configuration loaded.  Target: {catalog}.{schema}")
print(f"Date range: {DATA_START} to {DATA_END}")
print(f"Members: {NUM_MEMBERS:,} | Providers: {NUM_PROVIDERS:,} | Claims: {NUM_CLAIM_HEADERS:,} | Detail lines: {NUM_CLAIM_DETAILS:,}")

Configuration loaded.  Target: aira_test.aibi_workshop_schema_day2
Date range: 2023-04-01 to 2026-04-13
Members: 20,000 | Providers: 1,000 | Claims: 100,000 | Detail lines: 300,000


In [0]:
# =====================================================================
#  SHARED REFERENCE DATA
#  Arrays used by 2+ table cells are defined ONCE here.
#  Table-specific arrays remain in their own cells.
# =====================================================================

# --- Names ---
FIRST_NAMES = [
    "James","Robert","John","Michael","David","William","Richard","Joseph","Thomas","Charles",
    "Christopher","Daniel","Matthew","Anthony","Mark","Steven","Paul","Andrew","Joshua","Kevin",
    "Mary","Patricia","Jennifer","Linda","Barbara","Elizabeth","Susan","Jessica","Sarah","Karen",
    "Lisa","Nancy","Betty","Margaret","Sandra","Ashley","Dorothy","Kimberly","Emily","Donna"]
LAST_NAMES = [
    "Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis","Rodriguez","Martinez",
    "Hernandez","Lopez","Gonzalez","Wilson","Anderson","Thomas","Taylor","Moore","Jackson","Martin",
    "Lee","Perez","Thompson","White","Harris","Sanchez","Clark","Ramirez","Lewis","Robinson"]

# --- Geography ---
STATES = [
    "CA","TX","FL","NY","PA","IL","OH","GA","NC","MI",
    "NJ","VA","WA","AZ","MA","TN","IN","MO","MD","WI",
    "CO","MN","SC","AL","LA","KY","OR","OK","CT","UT"]

# --- Source systems ---
SRC_CODES = ["QNXT","FACETS","AMISYS"]
SRC_NAMES = ["QNXT Claims System","FACETS Admin System","AMISYS Legacy"]

# --- Lines of business ---
LOB_CODES = ["COM","MCR","MCD","MMP","EXC"]
LOB_NAMES = ["Commercial","Medicare","Medicaid","Medicare-Medicaid Plan","Exchange"]
LOB_W     = [45, 25, 15, 5, 10]

# --- Provider specialties & types ---
SPECIALTIES = [
    "Family Medicine","Internal Medicine","Pediatrics","Obstetrics and Gynecology",
    "Cardiology","Orthopedic Surgery","Dermatology","Psychiatry","Neurology",
    "Gastroenterology","Pulmonology","Endocrinology","Oncology","Urology",
    "Ophthalmology","Otolaryngology","General Surgery","Emergency Medicine",
    "Anesthesiology","Radiology","Pathology","Physical Medicine","Nephrology",
    "Rheumatology","Allergy and Immunology"]
PROV_TYPES = ["MD","DO","NP","PA"]
PROV_TYPE_W = [55, 15, 20, 10]

# --- Plan codes ---
PLAN_CODES = ["HMO001","PPO001","EPO001","POS001","HDHP01","HMO002","PPO002"]

# --- Claim types ---
CLAIM_TYPES   = ["P","I"]   # Professional, Institutional
CLAIM_TYPE_W  = [85, 15]

# --- ICD-10 diagnosis codes ---
ICD10_CODES = [
    "I10","E11.9","J06.9","M54.5","Z00.00","J20.9","E78.5","Z23","K21.0","F32.9",
    "J45.909","M79.3","E03.9","N39.0","R10.9","G43.909","J02.9","L30.9","F41.1","R05.9",
    "M25.50","E66.01","Z12.31","J18.9","I25.10","K58.9","R51.9","N40.0","E55.9","G47.00",
    "M17.11","J44.1","Z96.641","I48.91","E11.65","D64.9","F17.210","R07.9","J30.1","L70.0",
    "Z87.891","M47.812","K76.0","I50.9","E87.6","R00.0","J40","N18.3","G89.29","Z79.899"]

print(f"Shared reference data loaded: {len(FIRST_NAMES)} first names, {len(LAST_NAMES)} last names, "
      f"{len(STATES)} states, {len(SPECIALTIES)} specialties, {len(ICD10_CODES)} ICD-10 codes")

Shared reference data loaded: 40 first names, 30 last names, 30 states, 25 specialties, 50 ICD-10 codes


In [0]:
# ---------------------------------------------------------------------------
# dim_address  (NUM_ADDRESSES rows)
# Uses shared: STATES, SRC_CODES, SRC_NAMES
# ---------------------------------------------------------------------------

# --- Table-specific reference data ---
CITIES = ["Los Angeles","Houston","Jacksonville","New York","Philadelphia","Chicago","Columbus",
          "Atlanta","Charlotte","Detroit","San Diego","Dallas","Miami","Buffalo","Pittsburgh",
          "Aurora","Cleveland","Augusta","Raleigh","Grand Rapids","San Jose","Austin","Tampa",
          "Rochester","Allentown","Rockford","Cincinnati","Macon","Greensboro","Warren"]

df_address = (
    dg.DataGenerator(spark, name="dim_address", rows=NUM_ADDRESSES, partitions=SPARK_PARTITIONS)
    .withColumn("address_key", LongType(), minValue=1, uniqueValues=NUM_ADDRESSES)
    .withColumn("entity_type_key", StringType(),
                values=["MEMBER","PROVIDER","FACILITY"], weights=[77, 8, 15], random=True)
    .withColumn("entity_dimension_key", LongType(), minValue=1, maxValue=NUM_ADDRESSES, random=True)
    .withColumn("address_type_code", StringType(),
                values=["HOME","MAIL","WORK","OFFICE"], weights=[40, 20, 10, 30], random=True)
    .withColumn("street_address_1", StringType(), template=r"\\v \\w St", random=True)
    .withColumn("street_address_2", StringType(),
                values=["Apt 101","Apt 202","Apt 303","Suite 100","Suite 200","Suite 300","Unit A","Unit B"],
                random=True, percentNulls=0.55)
    .withColumn("city", StringType(), values=CITIES, random=True)
    .withColumn("state", StringType(), values=STATES, random=True)
    .withColumn("zip_code", StringType(), template=r"\\d{5}", random=True)
    .withColumn("country_code", StringType(), values=["US"])
    .withColumn("county", StringType(), expr="concat(city, ' County')")
    .withColumn("is_active", BooleanType(), values=[True])
    .withColumn("valid_from_date", TimestampType(), begin="2023-04-01 00:00:00", end=f"{DATA_START} 00:00:00", random=True)
    .withColumn("valid_to_date", TimestampType(), percentNulls=1.0)
    .withColumn("source_system_code", StringType(), values=SRC_CODES, random=True)
    .withColumn("source_system_name", StringType(), values=SRC_NAMES, random=True)
    .withColumn("record_hash", StringType(), template=r"\\w{32}", random=True)
    .withColumn("created_at", TimestampType(), begin="2023-04-01 00:00:00", end=f"{DATA_START} 00:00:00", random=True)
    .withColumn("updated_at", TimestampType(), expr="created_at")
    .build()
)

df_address.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.dim_address")
print(f"dim_address: {df_address.count():,} rows written")

dim_address: 22,000 rows written


In [0]:
# ---------------------------------------------------------------------------
# dim_member  (NUM_MEMBERS rows)
# Age-aware LOB: seniors→Medicare, children→Medicaid/Commercial
# Uses shared: FIRST_NAMES, LAST_NAMES, STATES, SRC_CODES/NAMES, LOB_CODES/NAMES/W
# ---------------------------------------------------------------------------

# --- Table-specific reference data ---
RACES = ["White","Black or African American","Asian","American Indian or Alaska Native",
         "Native Hawaiian or Other Pacific Islander","Two or More Races"]
RACE_W = [58, 13, 6, 1, 1, 3]
ETHNICITIES = ["Not Hispanic or Latino","Hispanic or Latino"]
MARITAL = ["Single","Married","Divorced","Widowed","Separated"]
MARITAL_W = [30, 48, 11, 6, 5]
REL_TYPES = ["Self","Spouse","Child","Domestic Partner","Other"]
REL_W = [55, 20, 20, 3, 2]
PROV_GROUPS = ["HealthFirst Medical Group","Unity Care Partners","Premier Health Network",
               "Community Health Alliance","Pacific Medical Associates"]

df_member_raw = (
    dg.DataGenerator(spark, name="dim_member", rows=NUM_MEMBERS, partitions=SPARK_PARTITIONS)
    .withColumn("member_sk", LongType(), minValue=1, uniqueValues=NUM_MEMBERS)
    .withColumn("mbr_source_member_id", StringType(), expr="concat('SRC', lpad(cast(member_sk as string), 7, '0'))")
    .withColumn("mbr_member_id", IntegerType(), minValue=100001, uniqueValues=NUM_MEMBERS)
    .withColumn("mbr_deers_beneficiary_id", StringType(),
                expr="concat('DEERS', lpad(cast(member_sk as string), 8, '0'))", percentNulls=0.7)
    .withColumn("mbr_deers_family_id", StringType(),
                expr="concat('FAM', lpad(cast(cast(member_sk/3 as int) as string), 6, '0'))", percentNulls=0.7)
    .withColumn("mbr_sponsor_ssn", StringType(), template=r"\\d{3}-\\d{2}-\\d{4}", random=True, percentNulls=0.7)
    .withColumn("mbr_current_pcp_eff_date", DateType(), begin=DATA_START, end=DATA_END, random=True)
    .withColumn("mbr_current_pcp_nbr", StringType(),
                expr=f"concat('PCP', lpad(cast(cast(rand()*{NUM_PROVIDERS} + 1 as int) as string), 5, '0'))")
    .withColumn("mbr_dob", DateType(), begin="1931-01-01", end="2026-01-01", random=True)
    .withColumn("mbr_race", StringType(), values=RACES, weights=RACE_W, random=True)
    .withColumn("mbr_sex", StringType(), values=["M","F"], weights=[49,51], random=True)
    .withColumn("mbr_ethnicity", StringType(), values=ETHNICITIES, weights=[81,19], random=True)
    .withColumn("mbr_first_name", StringType(), values=FIRST_NAMES, random=True)
    .withColumn("mbr_middle_name", StringType(), values=list("ABCDEFGHIJKLMNOPQRSTUVWXYZ"), random=True)
    .withColumn("mbr_last_name", StringType(), values=LAST_NAMES, random=True)
    .withColumn("mbr_full_name", StringType(), expr="concat(mbr_first_name, ' ', mbr_middle_name, '. ', mbr_last_name)")
    .withColumn("mbr_marital_status", StringType(), values=MARITAL, weights=MARITAL_W, random=True)
    .withColumn("mbr_deceased_date", DateType(), begin=DATA_START, end=DATA_END, random=True, percentNulls=0.98)
    .withColumn("mbr_email", StringType(),
                expr="concat(lower(mbr_first_name), '.', lower(mbr_last_name), cast(member_sk as string), '@example.com')")
    .withColumn("mbr_phone_nbr", StringType(), template=r"(\\d{3}) \\d{3}-\\d{4}", random=True)
    .withColumn("mbr_mobile_nbr", StringType(), template=r"(\\d{3}) \\d{3}-\\d{4}", random=True, percentNulls=0.3)
    .withColumn("mbr_work_nbr", StringType(), template=r"(\\d{3}) \\d{3}-\\d{4}", random=True, percentNulls=0.6)
    .withColumn("mbr_sub_group_cd", StringType(), values=["SG01","SG02","SG03","SG04"], random=True)
    .withColumn("mbr_idcard_issue_date", DateType(), begin="2023-10-01", end=DATA_START, random=True)
    .withColumn("mbr_line_of_business", StringType(), values=LOB_CODES, weights=LOB_W, random=True)
    .withColumn("mbr_text_opt_in", StringType(), values=["Y","N"], random=True)
    .withColumn("mbr_current_riders", StringType(), values=["Dental","Vision","Dental+Vision"], random=True, percentNulls=0.5)
    .withColumn("mbr_authorized_rep", StringType(), values=["Y","N"], random=True)
    .withColumn("mbr_alt_sub_nbr", StringType(), expr="concat('ALT', lpad(cast(member_sk as string), 6, '0'))", percentNulls=0.8)
    .withColumn("mbr_relationship_type", StringType(), values=REL_TYPES, weights=REL_W, random=True)
    .withColumn("mbr_salary_tier", StringType(), values=["Tier1","Tier2","Tier3","Tier4"], random=True, percentNulls=0.45)
    .withColumn("mbr_pcp_auto_assigned", StringType(), values=["Y","N"], random=True)
    .withColumn("mbr_secured_flag", StringType(), values=["N","Y"], weights=[75,25], random=True)
    .withColumn("mbr_pharmacy_discount_flag", StringType(), values=["Y","N"], random=True)
    .withColumn("mbr_employee_id", StringType(), expr="concat('EMP', lpad(cast(member_sk as string), 6, '0'))", percentNulls=0.45)
    .withColumn("mbr_line_of_business_name", StringType(), values=LOB_NAMES, weights=LOB_W, random=True)
    .withColumn("mbr_responsible_party_id", StringType(), expr="concat('RP', lpad(cast(member_sk as string), 6, '0'))", percentNulls=0.55)
    .withColumn("mbr_responsible_party_name", StringType(), values=FIRST_NAMES, random=True, percentNulls=0.55)
    .withColumn("mbr_deceased_flag", StringType(), values=["N","Y"], weights=[98,2], random=True)
    .withColumn("mbr_pcp_lock_in_indicator", StringType(), values=["Y","N"], random=True)
    .withColumn("mbr_alt_person_nbr", StringType(), expr="concat('APN', lpad(cast(member_sk as string), 6, '0'))", percentNulls=0.85)
    .withColumn("mbr_state", StringType(), values=STATES, random=True)
    .withColumn("mbr_zip_code", StringType(), template=r"\\d{5}", random=True)
    .withColumn("mbr_pcp_lock_in_type", StringType(), values=["Hard","Soft"], random=True, percentNulls=0.4)
    .withColumn("mbr_provider_group_name", StringType(), values=PROV_GROUPS, random=True)
    .withColumn("mbr_name_suffix", StringType(), values=["Jr.","Sr.","III"], random=True, percentNulls=0.85)
    .withColumn("mbr_sub_flag", IntegerType(), values=[0, 1], weights=[45, 55], random=True)
    .withColumn("mbr_sub_ssn", StringType(), template=r"\\d{3}-\\d{2}-\\d{4}", random=True, percentNulls=0.45)
    .withColumn("mbr_extract_date", DateType(), values=[DATA_END])
    .withColumn("mbr_medicaid_case_nbr", StringType(),
                expr="concat('MCD', cast(cast(rand()*900000+100000 as int) as string))", percentNulls=0.85)
    .withColumn("mbr_relationship_ind", StringType(), values=["S","S","C","D","O"], random=True)
    .withColumn("source_system_code", StringType(), values=SRC_CODES, random=True)
    .withColumn("source_system_name", StringType(), values=SRC_NAMES, random=True)
    .withColumn("record_hash", StringType(), template=r"\\w{32}", random=True)
    .withColumn("created_at", TimestampType(), begin="2023-04-01 00:00:00", end=f"{DATA_START} 00:00:00", random=True)
    .withColumn("updated_at", TimestampType(), expr="created_at")
    .build()
)

# Post-process: derive is_active and end_date from mbr_deceased_flag
df_member_raw = (
    df_member_raw
    .withColumn("is_active",
        F.when(F.col("mbr_deceased_flag") == "Y", F.lit(False)).otherwise(F.lit(True)))
    .withColumn("end_date",
        F.when(F.col("mbr_deceased_flag") == "Y", F.col("mbr_deceased_date").cast(TimestampType()))
         .otherwise(F.lit(None).cast(TimestampType())))
)

# Post-process: make LOB age-aware  (seniors→MCR, children→MCD/COM)
df_member = df_member_raw.withColumn(
    "age", F.floor(F.datediff(F.lit(DATA_END), F.col("mbr_dob")) / 365.25)
).withColumn(
    "mbr_line_of_business",
    F.when(F.col("age") >= 65, F.lit("MCR"))
     .when(F.col("age") < 18, F.when(F.rand() < 0.6, F.lit("COM")).otherwise(F.lit("MCD")))
     .otherwise(F.col("mbr_line_of_business"))
).withColumn(
    "mbr_line_of_business_name",
    F.when(F.col("mbr_line_of_business") == "COM", F.lit("Commercial"))
     .when(F.col("mbr_line_of_business") == "MCR", F.lit("Medicare"))
     .when(F.col("mbr_line_of_business") == "MCD", F.lit("Medicaid"))
     .when(F.col("mbr_line_of_business") == "MMP", F.lit("Medicare-Medicaid Plan"))
     .otherwise(F.lit("Exchange"))
).drop("age")

df_member.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.dim_member")
print(f"dim_member: {df_member.count():,} rows written")

dim_member: 20,000 rows written


In [0]:
# ---------------------------------------------------------------------------
# dim_member_history  (NUM_HISTORY_VERSIONS rows)
# Derives FROM dim_member: cross-join each member with random version
# count, then mutate PCP / state to simulate SCD2 attribute changes.
# Uses shared: STATES
# ---------------------------------------------------------------------------
from pyspark.sql.window import Window

# Read the already-written dim_member table (guaranteed matching types)
df_mbr = spark.table(f"{catalog}.{schema}.dim_member")

# Generate a version-count per member (1–4 versions, weighted toward 2)
versions_df = (
    dg.DataGenerator(spark, name="versions", rows=NUM_MEMBERS, partitions=SPARK_PARTITIONS)
    .withColumn("member_sk", LongType(), minValue=1, uniqueValues=NUM_MEMBERS)
    .withColumn("num_versions", IntegerType(), values=[1,2,3,4], weights=[40,30,20,10], random=True)
    .build()
)

# Explode into one row per version
df_exploded = (
    versions_df
    .select("member_sk",
            F.explode(F.sequence(F.lit(1), F.col("num_versions"))).alias("version_idx"))
)

# Join to dim_member and simulate attribute changes per version
states_col = F.array([F.lit(s) for s in STATES])

df_history = (
    df_exploded.join(df_mbr, "member_sk")
    .withColumn("mbr_history_sk", F.monotonically_increasing_id() + 1)
    # Simulate PCP change in later versions
    .withColumn("mbr_current_pcp_nbr",
        F.when(F.col("version_idx") > 1,
               F.concat(F.lit("PCP"), F.lpad(F.floor(F.rand() * NUM_PROVIDERS + 1).cast("string"), 5, "0")))
         .otherwise(F.col("mbr_current_pcp_nbr")))
    .withColumn("mbr_current_pcp_eff_date",
        F.when(F.col("version_idx") > 1,
               F.date_add(F.lit(DATA_START), (F.rand() * 700).cast("int")))
         .otherwise(F.col("mbr_current_pcp_eff_date")))
    # Simulate state change in ~30% of later versions
    .withColumn("mbr_state",
        F.when((F.col("version_idx") > 1) & (F.rand() < 0.3),
               states_col.getItem((F.rand() * len(STATES)).cast("int")))
         .otherwise(F.col("mbr_state")))
    # Version timestamps
    .withColumn("valid_from_date",
        F.to_timestamp(F.date_add(F.lit(DATA_START), ((F.col("version_idx") - 1) * (F.rand() * 200 + 50)).cast("int"))))
    .withColumn("valid_to_date", F.lit(None).cast(TimestampType()))  # placeholder
    .withColumn("created_at", F.col("valid_from_date"))
    .withColumn("updated_at", F.col("valid_from_date"))
    .withColumn("record_hash", F.md5(F.concat(F.col("member_sk").cast("string"), F.col("version_idx").cast("string"))))
    .drop("version_idx", "num_versions")
)

# Mark latest version per member as is_active, set valid_to_date on older versions
w = Window.partitionBy("member_sk").orderBy(F.col("valid_from_date").desc())
df_history = (
    df_history
    .withColumn("_rn", F.row_number().over(w))
    .withColumn("is_active", F.col("_rn") == 1)
    .withColumn("valid_to_date",
        F.when(F.col("_rn") > 1,
               F.lead("valid_from_date").over(Window.partitionBy("member_sk").orderBy("valid_from_date")))
         .otherwise(F.lit(None).cast(TimestampType())))
    .drop("_rn")
)

df_history.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.dim_member_history")
print(f"dim_member_history: {df_history.count():,} rows written")

/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/column.py:402: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


dim_member_history: 40,109 rows written


In [0]:
# ---------------------------------------------------------------------------
# dim_member_identifier  (NUM_MEMBER_IDS rows)
# Uses shared: SRC_CODES, SRC_NAMES
# ---------------------------------------------------------------------------

# --- Table-specific reference data ---
ID_TYPES = ["DEERS_BEN","MEDICAID_ID","MEDICARE_HIC","SSN","ALTERNATE"]

df_member_id = (
    dg.DataGenerator(spark, name="dim_member_identifier", rows=NUM_MEMBER_IDS, partitions=SPARK_PARTITIONS)
    .withColumn("mbr_identifier_sk", LongType(), minValue=1, uniqueValues=NUM_MEMBER_IDS)
    .withColumn("member_sk", LongType(), minValue=1, maxValue=NUM_MEMBERS, random=True)
    .withColumn("id_type", StringType(), values=ID_TYPES, random=True)
    .withColumn("id_value", StringType(),
                expr="case "
                     "when id_type = 'SSN' then concat(cast(cast(rand()*900+100 as int) as string), '-', cast(cast(rand()*90+10 as int) as string), '-', cast(cast(rand()*9000+1000 as int) as string)) "
                     "when id_type = 'DEERS_BEN' then concat('DEERS', lpad(cast(member_sk as string), 8, '0')) "
                     "when id_type = 'MEDICAID_ID' then concat('MCD', cast(cast(rand()*900000+100000 as int) as string)) "
                     "when id_type = 'MEDICARE_HIC' then concat('HIC', cast(cast(rand()*900000000+100000000 as int) as string)) "
                     "else concat('ALT', cast(cast(rand()*900000+100000 as int) as string)) end")
    .withColumn("source_system_code", StringType(), values=SRC_CODES, random=True)
    .withColumn("source_system_name", StringType(), values=SRC_NAMES, random=True)
    .withColumn("valid_to_date", TimestampType(), percentNulls=1.0)
    .withColumn("valid_from_date", TimestampType(), begin="2023-04-01 00:00:00", end=f"{DATA_START} 00:00:00", random=True)
    .withColumn("is_active", BooleanType(), values=[True])
    .withColumn("record_hash", StringType(), template=r"\\w{32}", random=True)
    .withColumn("created_at", TimestampType(), expr="valid_from_date")
    .withColumn("updated_at", TimestampType(), expr="valid_from_date")
    .build()
)

df_member_id.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.dim_member_identifier")
print(f"dim_member_identifier: {df_member_id.count():,} rows written")

dim_member_identifier: 40,000 rows written


In [0]:
# ---------------------------------------------------------------------------
# dim_provider  (NUM_PROVIDERS rows)
# Uses shared: FIRST_NAMES, LAST_NAMES, SPECIALTIES, PROV_TYPES, PROV_TYPE_W, SRC_CODES
# ---------------------------------------------------------------------------

df_provider_raw = (
    dg.DataGenerator(spark, name="dim_provider", rows=NUM_PROVIDERS, partitions=SPARK_PARTITIONS)
    .withColumn("provider_sk", LongType(), minValue=1, uniqueValues=NUM_PROVIDERS)
    .withColumn("assigned_provider_sk", LongType(), expr="provider_sk", percentNulls=0.6)
    .withColumn("source_provider_id", StringType(), expr="concat('PROV', lpad(cast(provider_sk as string), 6, '0'))")
    .withColumn("provider_npi", StringType(),
                expr="concat('1', cast(cast(rand()*900000000+100000000 as int) as string))")
    .withColumn("provider_tax_id", StringType(),
                expr="concat(cast(cast(rand()*90+10 as int) as string), '-', cast(cast(rand()*9000000+1000000 as int) as string))")
    .withColumn("provider_first", StringType(), values=FIRST_NAMES, random=True)
    .withColumn("provider_last", StringType(), values=LAST_NAMES, random=True)
    .withColumn("provider_type", StringType(), values=PROV_TYPES, weights=PROV_TYPE_W, random=True)
    .withColumn("provider_address_sk", LongType(), expr=f"cast({NUM_MEMBERS} + provider_sk as bigint)")
    .withColumn("specialty", StringType(), values=SPECIALTIES, random=True)
    .withColumn("affiliation_id", StringType(),
                expr="concat('AFF', lpad(cast(cast(rand()*50+1 as int) as string), 3, '0'))")
    .withColumn("valid_from_date", TimestampType(), begin="2022-04-01 00:00:00", end=f"{DATA_START} 00:00:00", random=True)
    .withColumn("valid_to_date", TimestampType(), percentNulls=1.0)
    .withColumn("is_active", BooleanType(), values=[True])
    .withColumn("source_system", StringType(), values=SRC_CODES, random=True)
    .withColumn("created_at", TimestampType(), expr="valid_from_date")
    .withColumn("updated_at", TimestampType(), expr="valid_from_date")
    .withColumn("last_update_dt", TimestampType(), expr="valid_from_date")
    .withColumn("record_hash", StringType(), template=r"\\w{32}", random=True)
    .build()
)

# Post-process: derive computed columns from staging columns, then drop them
# (dbldatagen expr= cannot reference columns defined with values=+weights=)
df_provider = (
    df_provider_raw
    .withColumn("provider_name",
        F.concat(F.lit("Dr. "), F.col("provider_first"), F.lit(" "),
                 F.col("provider_last"), F.lit(", "), F.col("provider_type")))
    .withColumn("pcp_flag",
        F.col("specialty").isin("Family Medicine", "Internal Medicine", "Pediatrics"))
    .drop("provider_first", "provider_last", "provider_type", "specialty")
)

df_provider.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.dim_provider")
print(f"dim_provider: {df_provider.count():,} rows written")

dim_provider: 1,000 rows written


In [0]:
# ---------------------------------------------------------------------------
# fact_member_enrollment  (NUM_ENROLLMENTS rows)
# Uses shared: SRC_CODES, LOB_CODES, LOB_W, PLAN_CODES
# ---------------------------------------------------------------------------

# --- Table-specific reference data ---
GROUP_NAMES = ["Federal Employees Group","State Employees Group","Large Employer Group A",
               "Large Employer Group B","Small Business Group","Individual Market",
               "Retiree Group","TRICARE Select","TRICARE Prime","Medicaid Expansion"]
SUBGROUP_NAMES = ["Standard Plan","Premium Plan","Basic Plan","High Deductible Plan","PPO Plan"]
TERM_REASONS = ["Voluntary Disenrollment","Non-Payment","Loss of Eligibility",
                "Transfer to Other Plan","Deceased","End of Coverage Period"]

df_enrollment = (
    dg.DataGenerator(spark, name="fact_member_enrollment", rows=NUM_ENROLLMENTS, partitions=SPARK_PARTITIONS)
    .withColumn("enrollment_sk", StringType(), expr="concat('ENR', lpad(cast(id as string), 8, '0'))")
    .withColumn("member_sk", LongType(), minValue=1, maxValue=NUM_MEMBERS, random=True)
    .withColumn("source_system", StringType(), values=SRC_CODES, random=True)
    .withColumn("src_member_business_id", IntegerType(), expr="cast(100000 + member_sk as int)")
    .withColumn("mbr_enr_source_member_id", StringType(), expr="concat('SRC', lpad(cast(member_sk as string), 7, '0'))")
    .withColumn("mbr_enr_insured_id", StringType(), expr="concat('INS', lpad(cast(member_sk as string), 7, '0'))")
    .withColumn("mbr_enr_contract_id", StringType(),
                expr="concat('CON', cast(cast(rand()*900000+100000 as int) as string))")
    .withColumn("mbr_enr_insured_code", StringType(), values=["01","02","03","18","19"], random=True)
    .withColumn("mbr_enr_effective_date", DateType(), begin=DATA_START, end=DATA_END, random=True)
    .withColumn("mbr_enr_insured_add_date", TimestampType(),
                expr="cast(date_sub(mbr_enr_effective_date, cast(rand()*30+1 as int)) as timestamp)")
    .withColumn("mbr_enr_insured_event_date", TimestampType(), expr="cast(mbr_enr_effective_date as timestamp)")
    .withColumn("mbr_enr_insured_event_id", StringType(), expr="concat('EVT', lpad(cast(id as string), 7, '0'))")
    .withColumn("mbr_enr_insured_event_code", StringType(), values=["NE","RE","TR","CH"], random=True)
    # Use expr= for weighted columns that are referenced by other expr= columns
    .withColumn("mbr_enr_status", StringType(),
                expr="element_at(array('Active','Active','Active','Active','Active','Active','Active','Active','Active','Active','Active','Active','Active','Active','Active','Terminated','Terminated','Terminated','Suspended','Pending'), cast(rand()*20+1 as int))")
    .withColumn("mbr_enr_insured_event_add_date", TimestampType(), expr="mbr_enr_insured_event_date")
    .withColumn("mbr_enr_plan_id", StringType(), values=PLAN_CODES, random=True)
    .withColumn("mbr_enr_line_of_business", StringType(),
                expr="element_at(array('COM','COM','COM','COM','COM','COM','COM','COM','COM','MCR','MCR','MCR','MCR','MCR','MCD','MCD','MCD','MMP','EXC','EXC'), cast(rand()*20+1 as int))")
    .withColumn("mbr_enr_line_of_business_id", StringType(),
                expr="case when mbr_enr_line_of_business='COM' then 'LOB001' "
                     "when mbr_enr_line_of_business='MCR' then 'LOB002' "
                     "when mbr_enr_line_of_business='MCD' then 'LOB003' "
                     "when mbr_enr_line_of_business='MMP' then 'LOB004' else 'LOB005' end")
    .withColumn("mbr_enr_group_name", StringType(), values=GROUP_NAMES, random=True)
    .withColumn("mbr_enr_subgroup_name", StringType(), values=SUBGROUP_NAMES, random=True)
    .withColumn("mbr_enr_termination_date", DateType(),
                expr="case when mbr_enr_status = 'Terminated' then date_add(mbr_enr_effective_date, cast(rand()*540+180 as int)) else null end")
    .withColumn("mbr_enr_termination_event_date", DateType(), expr="mbr_enr_termination_date")
    .withColumn("mbr_enr_termination_reason", StringType(), values=TERM_REASONS, random=True, percentNulls=0.75)
    .withColumn("mbr_enr_product_id", StringType(),
                expr="concat('PROD', lpad(cast(cast(rand()*20+1 as int) as string), 3, '0'))")
    .withColumn("mbr_enr_group_ck", StringType(),
                expr="concat('GCK', lpad(cast(cast(rand()*50+1 as int) as string), 4, '0'))")
    .withColumn("mbr_enr_group_code", StringType(),
                expr="concat('G', cast(cast(rand()*900+100 as int) as string))")
    .withColumn("mbr_enr_subgroup_ck", StringType(),
                expr="concat('SGCK', lpad(cast(cast(rand()*100+1 as int) as string), 4, '0'))")
    .withColumn("mbr_enr_subgroup_code", StringType(),
                expr="concat('SG', cast(cast(rand()*90+10 as int) as string))")
    .withColumn("id_value", StringType(), expr="mbr_enr_insured_id")
    .withColumn("id_type", StringType(), values=["INSURED_ID"])
    .withColumn("payload_hash", StringType(), template=r"\\w{32}", random=True)
    .build()
)

# Post-process: derive is_active from mbr_enr_status
df_enrollment = df_enrollment.withColumn(
    "is_active", F.col("mbr_enr_status") == F.lit("Active")
)

df_enrollment.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.fact_member_enrollment")
print(f"fact_member_enrollment: {df_enrollment.count():,} rows written")

fact_member_enrollment: 20,000 rows written


In [0]:
# ---------------------------------------------------------------------------
# fact_claim_header  (NUM_CLAIM_HEADERS rows)
# Uses shared: CLAIM_TYPES, CLAIM_TYPE_W, LAST_NAMES, SPECIALTIES,
#              PROV_TYPES, PROV_TYPE_W, LOB_CODES, LOB_W, PLAN_CODES,
#              SRC_CODES, SRC_NAMES, ICD10_CODES
# ---------------------------------------------------------------------------

# --- Table-specific reference data ---
ADMIT_TYPES = ["1","2","3","4","5","9"]; AT_W = [30,25,30,5,2,8]
ADMIT_SRC   = ["1","2","4","5","7","8"]; AS_W = [40,15,10,25,5,5]
BILL_TYPES  = ["131","111","121","141","711","731","771","851"]
DRG_CODES   = ["470","871","392","291","690","378","191","065","194","641",
               "193","812","683","189","292","309","313","638","310","247"]

# ---- Builder: generate base columns only (no cross-referencing expr) ----
gen_hdr = (
    dg.DataGenerator(spark, name="fact_claim_header", rows=NUM_CLAIM_HEADERS, partitions=SPARK_PARTITIONS)
    .withColumn("clm_header_sk", LongType(), minValue=1, uniqueValues=NUM_CLAIM_HEADERS)
    .withColumn("clm_id", IntegerType(), expr="cast(100000 + clm_header_sk as int)")
    .withColumn("clm_claim_id", StringType(), expr="concat('CLM', lpad(cast(clm_header_sk as string), 8, '0'))")
    .withColumn("clm_original_source_claim_id", StringType(), expr="concat('SRC', lpad(cast(clm_header_sk as string), 8, '0'))")
    .withColumn("clm_original_batch_nbr", StringType(), template=r"BATCH2025\\d{4}", random=True)
    .withColumn("clm_patient_control_nbr", StringType(), expr="concat('PCN', cast(cast(rand()*900000+100000 as int) as string))")
    .withColumn("clm_document_adj_control_nbr", StringType(), expr="concat('DAC', cast(cast(rand()*900000+100000 as int) as string))", percentNulls=0.9)
    .withColumn("clm_claim_type", StringType(), expr="case when rand() < 0.85 then 'P' else 'I' end")
    .withColumn("clm_bill_type", StringType(), values=BILL_TYPES, random=True)
    .withColumn("clm_authorization_nbr", StringType(), expr="concat('AUTH', cast(cast(rand()*900000+100000 as int) as string))", percentNulls=0.7)
    .withColumn("clm_ext_authorization_nbr", StringType(), percentNulls=1.0)
    .withColumn("clm_add_date", DateType(), begin=DATA_START, end=DATA_END, random=True)
    .withColumn("clm_admission_hour", IntegerType(), minValue=0, maxValue=23, random=True)
    .withColumn("clm_discharge_hour", IntegerType(), minValue=0, maxValue=23, random=True)
    .withColumn("clm_admission_type", StringType(), values=ADMIT_TYPES, weights=AT_W, random=True)
    .withColumn("clm_admission_source", StringType(), values=ADMIT_SRC, weights=AS_W, random=True)
    .withColumn("clm_member_sk", LongType(), minValue=1, maxValue=NUM_MEMBERS, random=True)
    .withColumn("clm_member_name", StringType(), values=LAST_NAMES, random=True)
    .withColumn("clm_member_group_nbr", StringType(), values=["SG01","SG02","SG03","SG04"], random=True)
    .withColumn("clm_member_subgroup_nbr", StringType(), values=["SUB01","SUB02","SUB03","SUB04","SUB05"], random=True)
    .withColumn("clm_plan_code", StringType(), values=PLAN_CODES, random=True)
    .withColumn("clm_line_of_business", StringType(), values=LOB_CODES, weights=LOB_W, random=True)
    .withColumn("clm_birth_weight", DecimalType(9,2), expr="cast(rand()*3.0 + 2.0 as decimal(9,2))", percentNulls=0.98)
    .withColumn("clm_accident_st", StringType(), values=["Y","N"], weights=[5,95], random=True, percentNulls=0.3)
    .withColumn("clm_attending_physician", StringType(), values=LAST_NAMES, random=True)
    .withColumn("clm_attending_physician_spec", StringType(), values=SPECIALTIES, random=True)
    .withColumn("clm_operating_provider_name", StringType(), values=LAST_NAMES, random=True)
    .withColumn("clm_submitting_provider", StringType(), values=LAST_NAMES, random=True)
    .withColumn("clm_submitting_provider_spec", StringType(), values=SPECIALTIES, random=True)
    .withColumn("clm_submitting_provider_type", StringType(), values=PROV_TYPES, weights=PROV_TYPE_W, random=True)
    .withColumn("clm_billing_provider", StringType(), values=LAST_NAMES, random=True)
    .withColumn("clm_referring_provider", StringType(), values=LAST_NAMES, random=True, percentNulls=0.6)
    .withColumn("clm_referring_provider_spec", StringType(), values=SPECIALTIES, random=True, percentNulls=0.6)
    .withColumn("clm_referring_provider_type", StringType(), values=PROV_TYPES, random=True, percentNulls=0.6)
    .withColumn("clm_is_par_submitting_provider", StringType(), values=["Y","N"], weights=[70,30], random=True)
    .withColumn("clm_is_par_referring_provider", BooleanType(), values=[True,False], weights=[70,30], random=True, percentNulls=0.6)
    .withColumn("clm_is_par_rendering_provider", BooleanType(), values=[True,False], weights=[80,20], random=True)
    .withColumn("clm_pcp_visit_flag", IntegerType(), values=[0,1], weights=[60,40], random=True)
    .withColumn("clm_pcp_inl_type", StringType(), values=["Standard","In-Lieu"], random=True, percentNulls=0.5)
    .withColumn("clm_pcp_wthld_pct", DecimalType(9,4), expr="cast(rand()*0.15 as decimal(9,4))", percentNulls=0.6)
    .withColumn("clm_service_fac_loc_name", StringType(),
                values=["General Hospital","Regional Medical Center","Community Hospital","University Hospital"], random=True)
    .withColumn("clm_admitting_diagnosis_code", StringType(), values=ICD10_CODES, random=True)
    .withColumn("clm_diagnosis_1", StringType(), values=ICD10_CODES, random=True)
    .withColumn("clm_diagnosis_1_icd_method", StringType(), values=["ICD10"])
    .withColumn("clm_diagnosis_1_poa", StringType(), values=["Y","N","U","W"], random=True)
    .withColumn("clm_service_type_code", StringType(), values=["01","02","03","04","05"], random=True)
    .withColumn("clm_cob_type", StringType(), values=["Primary","Secondary"], random=True, percentNulls=0.5)
    .withColumn("clm_claim_timely_filing", StringType(), values=["Y","N"], weights=[90,10], random=True)
    .withColumn("clm_accept_assignment_indicator", StringType(), values=["Y","N"], weights=[85,15], random=True)
    .withColumn("clm_add_user", StringType(), values=["SYSTEM"])
    .withColumn("clm_update_user", StringType(), values=["SYSTEM"])
    .withColumn("clm_user_id", StringType(), values=["SYS"])
    .withColumn("clm_orig_source", StringType(), values=SRC_CODES, random=True)
    .withColumn("is_active", BooleanType(), values=[True])
    .withColumn("end_date", TimestampType(), percentNulls=1.0)
    .withColumn("record_hash", StringType(), template=r"\\w{32}", random=True)
    .withColumn("source_system_code", StringType(), values=SRC_CODES, random=True)
    .withColumn("source_system_name", StringType(), values=SRC_NAMES, random=True)
)

# Diagnosis codes 2–25 (base values only, no conditional expr)
for dx_i in range(2, 26):
    null_pct = min(0.95, 0.30 + (dx_i - 2) * 0.03)
    gen_hdr = (
        gen_hdr
        .withColumn(f"clm_diagnosis_{dx_i}", StringType(), values=ICD10_CODES, random=True, percentNulls=null_pct)
        .withColumn(f"clm_diagnosis_{dx_i}_icd_method", StringType(), values=["ICD10"], percentNulls=null_pct)
        .withColumn(f"clm_diagnosis_{dx_i}_poa", StringType(), values=["Y","N","U","W"], random=True)
    )

# Surgical codes 1–6 (base values only)
for s_i in range(1, 7):
    null_pct = 0.70 + (s_i - 1) * 0.05
    gen_hdr = (
        gen_hdr
        .withColumn(f"clm_surgical_{s_i}", StringType(), values=["0210","0DBN","0SB0","0UT9","0W3N"], random=True, percentNulls=null_pct)
        .withColumn(f"clm_surgical_icd_method_{s_i}", StringType(), values=["ICD10PCS"], percentNulls=null_pct)
    )

gen_hdr = gen_hdr.withColumn("clm_drg_code", StringType(), values=DRG_CODES, random=True)

df_raw = gen_hdr.build()

# ---- Post-processing: all conditional and derived columns ----
is_inst = F.col("clm_claim_type") == "I"

df_claim_header = (
    df_raw
    # Institutional-only columns: null for professional claims
    .withColumn("clm_bill_type", F.when(is_inst, F.col("clm_bill_type")))
    .withColumn("clm_admit_date", F.when(is_inst, F.col("clm_add_date")))
    .withColumn("clm_discharge_date", F.when(is_inst, F.date_add(F.col("clm_add_date"), (F.rand()*13+1).cast("int"))))
    .withColumn("clm_claim_thru_date", F.coalesce(F.col("clm_discharge_date"), F.col("clm_add_date")))
    .withColumn("clm_admission_hour", F.when(is_inst, F.col("clm_admission_hour")))
    .withColumn("clm_discharge_hour", F.when(is_inst, F.col("clm_discharge_hour")))
    .withColumn("clm_admission_type", F.when(is_inst, F.col("clm_admission_type")))
    .withColumn("clm_admission_source", F.when(is_inst, F.col("clm_admission_source")))
    .withColumn("clm_covered_days", F.when(is_inst, F.datediff(F.col("clm_discharge_date"), F.col("clm_admit_date"))))
    .withColumn("clm_drg_code", F.when(is_inst, F.col("clm_drg_code")))
    .withColumn("clm_admitting_diagnosis_code", F.when(is_inst, F.col("clm_admitting_diagnosis_code")))
    .withColumn("clm_admitting_diagnosis_method", F.when(is_inst, F.lit("ICD10")))
    .withColumn("clm_service_fac_loc_name", F.when(is_inst, F.col("clm_service_fac_loc_name")))
    .withColumn("clm_service_fac_npi", F.when(is_inst,
        F.concat(F.lit("1"), F.floor(F.rand()*900000000+100000000).cast("string"))))
    .withColumn("clm_service_facility_address_sk", F.when(is_inst,
        (F.lit(NUM_MEMBERS + NUM_PROVIDERS) + (F.rand()*1000+1).cast("long"))))
    # Provider name concats
    .withColumn("clm_attending_physician", F.concat(F.lit("Dr. "), F.col("clm_attending_physician")))
    .withColumn("clm_submitting_provider", F.concat(F.lit("Dr. "), F.col("clm_submitting_provider")))
    .withColumn("clm_billing_provider", F.concat(F.lit("Dr. "), F.col("clm_billing_provider")))
    .withColumn("clm_referring_provider",
        F.when(F.col("clm_referring_provider").isNotNull(), F.concat(F.lit("Dr. "), F.col("clm_referring_provider"))))
    .withColumn("clm_operating_provider_name",
        F.when(is_inst, F.concat(F.lit("Dr. "), F.col("clm_operating_provider_name"))))
    .withColumn("clm_operating_provider_npi", F.when(is_inst,
        F.concat(F.lit("1"), F.floor(F.rand()*900000000+100000000).cast("string"))))
    # Member-derived fields
    .withColumn("clm_member_nbr", F.concat(F.lit("SRC"), F.lpad(F.col("clm_member_sk").cast("string"), 7, "0")))
    .withColumn("clm_assigned_pcp", F.concat(F.lit("PCP"), F.lpad(F.floor(F.rand()*NUM_PROVIDERS+1).cast("string"), 5, "0")))
    .withColumn("clm_assigned_provider_site", F.concat(F.lit("SITE"), F.lpad(F.floor(F.rand()*100+1).cast("string"), 3, "0")))
    .withColumn("clm_insurance_id", F.concat(F.lit("INS"), F.lpad(F.col("clm_member_sk").cast("string"), 7, "0")))
    .withColumn("clm_member_contract_id", F.concat(F.lit("CON"), F.floor(F.rand()*900000+100000).cast("string")))
    .withColumn("clm_onset_of_illness_date",
        F.when(F.rand() < 0.3, F.date_sub(F.col("clm_add_date"), (F.rand()*7).cast("int")).cast("string")))
    # Diagnosis 1 POA (institutional only)
    .withColumn("clm_diagnosis_1_poa", F.when(is_inst, F.col("clm_diagnosis_1_poa")))
    # Date-derived metadata
    .withColumn("clm_update_date", F.date_add(F.col("clm_add_date"), (F.rand()*30+1).cast("int")))
    .withColumn("clm_extract_date", F.col("clm_update_date").cast(TimestampType()))
    .withColumn("clm_last_updated_date", F.col("clm_update_date"))
    .withColumn("clm_original_insert_date", F.col("clm_add_date"))
    .withColumn("last_updated_date", F.col("clm_update_date"))
    .withColumn("created_at", F.col("clm_add_date").cast(TimestampType()))
    .withColumn("updated_at", F.col("clm_extract_date"))
)

# Null out diagnosis POA and surgical codes for professional claims
for dx_i in range(2, 26):
    df_claim_header = (
        df_claim_header
        .withColumn(f"clm_diagnosis_{dx_i}_icd_method",
            F.when(F.col(f"clm_diagnosis_{dx_i}").isNotNull(), F.lit("ICD10")))
        .withColumn(f"clm_diagnosis_{dx_i}_poa",
            F.when(is_inst & F.col(f"clm_diagnosis_{dx_i}").isNotNull(), F.col(f"clm_diagnosis_{dx_i}_poa")))
    )
for s_i in range(1, 7):
    df_claim_header = (
        df_claim_header
        .withColumn(f"clm_surgical_{s_i}", F.when(is_inst, F.col(f"clm_surgical_{s_i}")))
        .withColumn(f"clm_surgical_icd_method_{s_i}",
            F.when(F.col(f"clm_surgical_{s_i}").isNotNull(), F.lit("ICD10PCS")))
    )

df_claim_header.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.fact_claim_header")
print(f"fact_claim_header: {df_claim_header.count():,} rows written")

fact_claim_header: 100,000 rows written


In [0]:
# ---------------------------------------------------------------------------
# fact_claim_detail  (NUM_CLAIM_DETAILS rows)
# Realistic financial amounts: inpatient >> outpatient
# Uses shared: CLAIM_TYPES, CLAIM_TYPE_W, LAST_NAMES, SPECIALTIES,
#              PROV_TYPES, PROV_TYPE_W, SRC_CODES
# ---------------------------------------------------------------------------

# --- Table-specific reference data ---
POS   = ["11","21","22","23","31","32","49","81"]; POS_W = [50,15,10,8,5,4,4,4]
REV_CODES = ["0100","0120","0250","0260","0270","0300","0320","0360","0370","0450",
             "0510","0636","0730","0942","0960","0981"]
CPT = [
    "99213","99214","99215","99203","99204","99395","99396","99397",
    "99283","99284","99285","36415","85025","80053","80061","81001",
    "71046","73030","72148","70553","93000","90837","90834","90847",
    "97110","97140","97530","97112","43239","45380","27447","29881",
    "10060","11102","17000","20610","64483","62323","99232","99233"]

# ---- Builder: base columns only ----
df_dtl_raw = (
    dg.DataGenerator(spark, name="fact_claim_detail", rows=NUM_CLAIM_DETAILS, partitions=SPARK_PARTITIONS)
    .withColumn("clm_dtl_claim_id", StringType(),
                expr=f"concat('CLM', lpad(cast(cast(rand()*{NUM_CLAIM_HEADERS}+1 as int) as string), 8, '0'))")
    .withColumn("clm_dtl_line_nbr", StringType(), expr="cast(cast(rand()*9+1 as int) as string)")
    .withColumn("clm_dtl_source_system", StringType(), values=SRC_CODES, random=True)
    .withColumn("clm_dtl_claim_type", StringType(), expr="case when rand() < 0.85 then 'P' else 'I' end")
    .withColumn("clm_dtl_specific_dos_date", DateType(), begin=DATA_START, end=DATA_END, random=True)
    .withColumn("clm_dtl_benefit_category", StringType(),
                values=["Medical","Behavioral Health","Pharmacy","Preventive"], weights=[60,15,10,15], random=True)
    .withColumn("clm_dtl_benefit_level", StringType(),
                values=["In-Network","Out-of-Network","Tier 1","Tier 2"], weights=[65,15,10,10], random=True)
    .withColumn("clm_dtl_interest_discount_flag", StringType(), values=["N"])
    .withColumn("clm_dtl_copay_reason", StringType(), values=["Standard","Specialist","ER","Urgent Care"], random=True, percentNulls=0.3)
    .withColumn("clm_dtl_st_cd", StringType(), values=["P","D","A"], weights=[75,15,10], random=True)
    .withColumn("clm_dtl_line_status", StringType(), values=["Paid","Denied","Pending","Adjusted"], weights=[75,10,5,10], random=True)
    .withColumn("clm_dtl_clean_claim_ind", StringType(), values=["Y","N"], weights=[80,20], random=True)
    .withColumn("clm_dtl_place_of_service", StringType(), values=POS, weights=POS_W, random=True)
    .withColumn("clm_dtl_fee_schedule_code", StringType(), values=["FS1","FS2","FS3","FS4"], random=True)
    .withColumn("clm_dtl_check_added_line_flag", StringType(), values=["Y","N"], random=True)
    .withColumn("clm_dtl_check_message", StringType(), percentNulls=1.0)
    .withColumn("clm_dtl_submitting_provider", StringType(), values=LAST_NAMES, random=True)
    .withColumn("clm_dtl_rendering_provider", StringType(), values=LAST_NAMES, random=True)
    .withColumn("clm_dtl_rendering_provider_type", StringType(), values=PROV_TYPES, weights=PROV_TYPE_W, random=True)
    .withColumn("clm_dtl_rendering_provider_spec", StringType(), values=SPECIALTIES, random=True)
    .withColumn("clm_dtl_participating_provider", StringType(), values=["Y","N"], weights=[70,30], random=True)
    .withColumn("clm_dtl_adjudication_status", StringType(), values=["Final","Interim","Reopened"], weights=[85,10,5], random=True)
    .withColumn("clm_dtl_procedure_code", StringType(), values=CPT, random=True)
    .withColumn("clm_dtl_procedure_modifier", StringType(), values=["25","59","76","77"], random=True, percentNulls=0.5)
    .withColumn("clm_dtl_modifier_2", StringType(), values=["TC","26"], random=True, percentNulls=0.7)
    .withColumn("clm_dtl_modifier_3", StringType(), percentNulls=1.0)
    .withColumn("clm_dtl_modifier_4", StringType(), percentNulls=1.0)
    .withColumn("clm_dtl_procedure_adj", StringType(), percentNulls=1.0)
    .withColumn("clm_dtl_revenue_code", StringType(), values=REV_CODES, random=True)
    .withColumn("clm_dtl_diagnosis_ind_1", StringType(), values=["01"])
    .withColumn("clm_dtl_diagnosis_ind_2", StringType(), values=["02"], percentNulls=0.5)
    .withColumn("clm_dtl_diagnosis_ind_3", StringType(), values=["03"], percentNulls=0.7)
    .withColumn("clm_dtl_diagnosis_ind_4", StringType(), values=["04"], percentNulls=0.9)
    .withColumn("clm_dtl_cob_rule", StringType(), values=["Standard"], percentNulls=0.92)
    .withColumn("clm_dtl_wrap_network", StringType(), values=["WrapNet1","WrapNet2"], random=True, percentNulls=0.7)
    .withColumn("clm_dtl_returned_ntwrk_repric", StringType(), percentNulls=1.0)
    .withColumn("clm_dtl_user_id", StringType(), values=["SYS"])
    .withColumn("is_active", BooleanType(), values=[True])
    .build()
)

# ---- Post-processing: all derived and conditional columns ----
is_inst = F.col("clm_dtl_claim_type") == "I"

df_claim_detail = (
    df_dtl_raw
    # Source-derived IDs
    .withColumn("clm_dtl_original_source_claim_id", F.concat(F.lit("SRC"), F.substring(F.col("clm_dtl_claim_id"), 4, 8)))
    .withColumn("clm_dtl_member_nbr_sk",
        F.concat(F.lit("SRC"), F.lpad(F.floor(F.rand()*NUM_MEMBERS+1).cast("string"), 7, "0")))
    # Date-derived fields
    .withColumn("clm_dtl_specific_dos_thru_date", F.col("clm_dtl_specific_dos_date"))
    .withColumn("clm_dtl_ap_posting_date", F.date_add(F.col("clm_dtl_specific_dos_date"), (F.rand()*25+20).cast("int")))
    .withColumn("clm_dtl_claim_receive_date",
        F.date_add(F.col("clm_dtl_specific_dos_date"), (F.rand()*14+1).cast("int")).cast(TimestampType()))
    .withColumn("clm_dtl_check_date",
        F.date_add(F.col("clm_dtl_specific_dos_date"), (F.rand()*45+15).cast("int")).cast(TimestampType()))
    .withColumn("clm_dtl_last_update_date", F.col("clm_dtl_check_date"))
    .withColumn("clmdetail_admit_dt", F.when(is_inst, F.col("clm_dtl_specific_dos_date").cast(TimestampType())))
    # Financial amounts: inpatient >> outpatient
    .withColumn("clm_dtl_billed_amt",
        F.when(is_inst, (F.rand()*49000+1000).cast("decimal(27,4)"))
         .otherwise((F.rand()*4950+50).cast("decimal(27,4)")))
    .withColumn("clm_dtl_allowed_amt", (F.col("clm_dtl_billed_amt") * (F.rand()*0.45+0.40)).cast("decimal(28,4)"))
    .withColumn("clm_dtl_deduct_amt", F.least(F.col("clm_dtl_allowed_amt") * F.rand() * 0.15, F.lit(2000)).cast("decimal(27,4)"))
    .withColumn("clm_dtl_copay_amt",
        F.element_at(F.array(*[F.lit(v).cast("decimal(27,4)") for v in [0,15,25,30,40,50,75,100,150,250]]),
                     (F.rand()*10+1).cast("int")))
    .withColumn("clm_dtl_co_insurance_amt",
        F.greatest(F.lit(0), (F.col("clm_dtl_allowed_amt") - F.col("clm_dtl_deduct_amt") - F.col("clm_dtl_copay_amt")) * F.rand() * 0.20).cast("decimal(27,4)"))
    .withColumn("clm_dtl_paid_amt",
        F.greatest(F.lit(0), F.col("clm_dtl_allowed_amt") - F.col("clm_dtl_deduct_amt") - F.col("clm_dtl_copay_amt") - F.col("clm_dtl_co_insurance_amt")).cast("decimal(28,4)"))
    .withColumn("clm_dtl_actual_paid_amt", F.col("clm_dtl_paid_amt").cast("decimal(38,4)"))
    .withColumn("clm_dtl_net_amt", (F.col("clm_dtl_paid_amt") * (1 - F.rand()*0.05)).cast("decimal(28,4)"))
    .withColumn("clm_dtl_not_covered_amt", (F.col("clm_dtl_billed_amt") - F.col("clm_dtl_allowed_amt")).cast("decimal(27,4)"))
    .withColumn("clm_dtl_other_adjustments_amt",
        F.when(F.rand() < 0.1, F.rand()*50).otherwise(F.lit(0)).cast("decimal(30,2)"))
    .withColumn("clm_dtl_cob_savings",
        F.when(F.rand() < 0.08, F.rand()*450+50).otherwise(F.lit(0)).cast("decimal(32,4)"))
    .withColumn("clm_dtl_oic_paid_amt",
        F.when(F.col("clm_dtl_cob_savings") > 0, F.rand()*280+20).otherwise(F.lit(0)).cast("decimal(19,4)"))
    .withColumn("clm_dtl_oic_allowed_amt", (F.col("clm_dtl_oic_paid_amt") * 1.2).cast("decimal(19,4)"))
    .withColumn("clm_dtl_interest_amt",
        F.when(F.rand() < 0.05, F.rand()*5).otherwise(F.lit(0)).cast("decimal(38,6)"))
    .withColumn("clm_dtl_prompt_pay_discount_amt",
        F.when(F.rand() < 0.1, F.rand()*45+5).otherwise(F.lit(0)).cast("decimal(20,4)"))
    .withColumn("clm_dtl_interest_discount_amt", F.lit(0).cast("decimal(38,6)"))
    .withColumn("clm_dtl_reason_code_sk",
        F.concat(F.lit("RC"), F.lpad(F.floor(F.rand()*50+1).cast("string"), 3, "0")))
    .withColumn("clm_dtl_authorization_nbr",
        F.when(F.rand() < 0.3, F.concat(F.lit("AUTH"), F.floor(F.rand()*900000+100000).cast("string"))))
    .withColumn("clm_dtl_check_nbr",
        F.when(F.col("clm_dtl_paid_amt") > 0,
               F.concat(F.lit("CHK"), F.floor(F.rand()*900000+100000).cast("string"))))
    # Provider name concats
    .withColumn("clm_dtl_submitting_provider", F.concat(F.lit("Dr. "), F.col("clm_dtl_submitting_provider")))
    .withColumn("clm_dtl_rendering_provider", F.concat(F.lit("Dr. "), F.col("clm_dtl_rendering_provider")))
    # Procedure qty
    .withColumn("clm_dtl_procedure_qty", (F.rand()*3+1).cast("decimal(15,3)"))
    # Institutional-only fields
    .withColumn("clm_dtl_revenue_code", F.when(is_inst, F.col("clm_dtl_revenue_code")))
    .withColumn("clm_dtl_paid_days",
        F.when(is_inst, (F.rand()*13+1).cast("int").cast("decimal(36,3)")).otherwise(F.lit(0).cast("decimal(36,3)")))
    .withColumn("clm_dtl_anesthesia_time_units", F.lit(0).cast("decimal(12,2)"))
    # Metadata
    .withColumn("clm_dtl_extract_date", F.col("clm_dtl_check_date"))
    .withColumn("updated_at", F.col("clm_dtl_check_date"))
    .withColumn("created_at", F.col("clm_dtl_claim_receive_date"))
    .drop("clm_dtl_claim_type")
)

df_claim_detail.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.fact_claim_detail")
print(f"fact_claim_detail: {df_claim_detail.count():,} rows written")

fact_claim_detail: 300,000 rows written


In [0]:
# ---------------------------------------------------------------------------
# Verification: row counts and sample data for all generated tables
# ---------------------------------------------------------------------------
tables = [
    "dim_address",
    "dim_member",
    "dim_member_history",
    "dim_member_identifier",
    "dim_provider",
    "fact_member_enrollment",
    "fact_claim_header",
    "fact_claim_detail",
]

results = []
for t in tables:
    fqn = f"{catalog}.{schema}.{t}"
    cnt = spark.table(fqn).count()
    results.append((fqn, cnt))

df_verify = spark.createDataFrame(results, ["table_name", "row_count"])
display(df_verify)

table_name,row_count
aira_test.aibi_workshop_schema_day2.dim_address,22000
aira_test.aibi_workshop_schema_day2.dim_member,20000
aira_test.aibi_workshop_schema_day2.dim_member_history,40109
aira_test.aibi_workshop_schema_day2.dim_member_identifier,40000
aira_test.aibi_workshop_schema_day2.dim_provider,1000
aira_test.aibi_workshop_schema_day2.fact_member_enrollment,20000
aira_test.aibi_workshop_schema_day2.fact_claim_header,100000
aira_test.aibi_workshop_schema_day2.fact_claim_detail,300000
